In [ ]:
# Cell 1: install/upgrade (safe on Colab)
!pip install -q -U datasets

# Cell 2: stream the dataset instead of downloading everything
from datasets import load_dataset
ds = load_dataset(
    "BanglishRev/bangla-english-and-code-mixed-ecommerce-review-dataset",
    streaming=True,
)
print(ds)                      # shows splits (probably just "train")

# Cell 3: look at the first few rows to see the real column names
split = list(ds.keys())[0]
for i, row in enumerate(ds[split]):
    print(row)
    if i == 4:
        break

README.md:   0%|          | 0.00/4.42k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/110 [00:00<?, ?it/s]

IterableDatasetDict({
    train: IterableDataset({
        features: ['image'],
        num_shards: 1
    })
})
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=512x682 at 0x78167CCB6E40>}
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=682x512 at 0x781658948550>}
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=512x682 at 0x781658948A50>}
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=682x512 at 0x78165A09A2C0>}
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=682x512 at 0x78165A09A650>}


In [ ]:
from huggingface_hub import HfApi

REPO = "BanglishRev/bangla-english-and-code-mixed-ecommerce-review-dataset"
info = HfApi().dataset_info(REPO, files_metadata=True)

# Print every file with size in MB, so we can spot the review text files
for f in sorted(info.siblings, key=lambda x: x.rfilename):
    size_mb = (f.size or 0) / 1e6
    print(f"{size_mb:10.1f} MB  {f.rfilename}")

       0.0 MB  .gitattributes
       0.0 MB  README.md
     492.9 MB  Review Images 1.zip
     481.7 MB  Review Images 10.zip
     453.8 MB  Review Images 100.zip
     446.4 MB  Review Images 101.zip
     446.7 MB  Review Images 102.zip
     470.3 MB  Review Images 103.zip
     460.7 MB  Review Images 104.zip
     461.3 MB  Review Images 105.zip
     455.0 MB  Review Images 106.zip
     473.2 MB  Review Images 107.zip
     494.0 MB  Review Images 108.zip
     171.2 MB  Review Images 109.zip
     487.1 MB  Review Images 11.zip
     483.4 MB  Review Images 12.zip
     485.0 MB  Review Images 13.zip
     474.2 MB  Review Images 14.zip
     486.5 MB  Review Images 15.zip
     498.6 MB  Review Images 16.zip
     480.2 MB  Review Images 17.zip
     481.9 MB  Review Images 18.zip
     513.1 MB  Review Images 19.zip
     482.0 MB  Review Images 2.zip
     492.3 MB  Review Images 20.zip
     485.3 MB  Review Images 21.zip
     483.3 MB  Review Images 22.zip
     488.7 MB  Review Images 23.zip
 

In [ ]:
from huggingface_hub import hf_hub_download

REPO = "BanglishRev/bangla-english-and-code-mixed-ecommerce-review-dataset"

# Downloads only this one file (~2 GB, takes a few minutes), cached afterwards
path = hf_hub_download(repo_id=REPO, filename="reviews v1.json", repo_type="dataset")
print(path)

# Peek at the first 3000 characters without loading the whole file
with open(path, "r", encoding="utf-8") as f:
    print(f.read(3000))

reviews v1.json: reconstructing file:   0%|          |  0.00B / 2.05GB            

reviews v1.json: downloading bytes:           |  0.00B            

/root/.cache/huggingface/hub/datasets--BanglishRev--bangla-english-and-code-mixed-ecommerce-review-dataset/snapshots/38c97cd4255799359b612bafb721e3c442bc0851/reviews v1.json
[
    {
        "Number Of Reviews": 2,
        "Reviews": [
            {
                "Buyer ID": "1797097",
                "Current Rating": 5,
                "Review Content": "i buy 3 box but 1 box not orginal product or 2 box i see orginal,, \n\u09b8\u09ac \u09ae\u09bf\u09b2\u09bf\u09df\u09c7 \u09ad\u09be\u09b2\u09cb\u0987, \u098f\u0987 \u09aa\u09be\u09b0\u09ab\u09bf\u0989\u09ae \u0997\u09c1\u09b2\u09be\u09b0 \u0997\u09cd\u09b0\u09be\u09a8 \u0986\u09ae\u09be\u09b0 \u0985\u09a8\u09c7\u0995  \u0985\u09a8\u09c7\u0995 \u09ad\u09be\u09b2\u09cb \u09b2\u09be\u0997\u09c7,, \n\u09a7\u09a8\u09cd\u09af\u09ac\u09be\u09a6 Teams",
                "Review Date": "2023-11-20",
                "Likes": 0,
                "Dislikes": 0,
                "Date Bought": "2023-11-15",
                "Reply": "Thanks for your

In [ ]:
!pip install -q ijson

import ijson, re, nltk, pandas as pd
nltk.download("words", quiet=True)
from nltk.corpus import words

SEED = 42
PATH = "/root/.cache/huggingface/hub/datasets--BanglishRev--bangla-english-and-code-mixed-ecommerce-review-dataset/snapshots/38c97cd4255799359b612bafb721e3c442bc0851/reviews v1.json"

BENGALI = re.compile(r"[\u0980-\u09FF]")          # any Bengali-script character
ENGLISH = set(w.lower() for w in words.words())   # English dictionary

def is_banglish(text):
    """Heuristic (same idea as the BanglishRev paper): Latin script only,
    >=3 tokens, and at least half the tokens are NOT English dictionary words."""
    if BENGALI.search(text):
        return False
    toks = re.findall(r"[a-z]+", text.lower())
    if len(toks) < 3:
        return False
    non_en = sum(t not in ENGLISH for t in toks)
    return non_en / len(toks) >= 0.5

rows, total = [], 0
with open(PATH, "rb") as f:
    for product in ijson.items(f, "item", use_float=True):   # streams one product at a time
        for r in product.get("Reviews", []):
            total += 1
            text = (r.get("Review Content") or "").strip()
            rating = r.get("Current Rating")
            if rating is not None and text and is_banglish(text):
                rows.append((text, int(rating)))

df = pd.DataFrame(rows, columns=["text", "rating"])
print("total reviews scanned:", total)
print("banglish candidates:", len(df))
print(df["rating"].value_counts().sort_index())

total reviews scanned: 3239574
banglish candidates: 156382
rating
1     15517
2      4707
3      8099
4     13294
5    114765
Name: count, dtype: int64


In [ ]:
# 1) Light privacy scrub: URLs and long digit strings (phone numbers)
df["text"] = (df["text"]
              .str.replace(r"https?://\S+", " ", regex=True)
              .str.replace(r"\d{7,}", " ", regex=True)
              .str.replace(r"\s+", " ", regex=True).str.strip())

# 2) Remove duplicates (many reviews are just "valo product"); prevents train/test leakage
df["key"] = df["text"].str.lower()
df = df.drop_duplicates("key").drop(columns="key")

# 3) Binary label from rating, same rule as the BanglishRev paper:
#    rating > 3 -> positive (1), rating <= 3 -> negative (0)
df["label"] = (df["rating"] > 3).astype(int)
print(df["label"].value_counts())

# 4) Seeded stratified sample. Natural data is ~heavily positive, so we cap it.
N_NEG, N_POS = 5000, 10000                     # adjust after seeing counts above
neg = df[df.label == 0].sample(min(N_NEG, (df.label == 0).sum()), random_state=SEED)
pos = df[df.label == 1].sample(min(N_POS, (df.label == 1).sum()), random_state=SEED)
sample = pd.concat([neg, pos]).sample(frac=1, random_state=SEED).reset_index(drop=True)

sample.to_csv("banglish_sample.csv", index=False, encoding="utf-8")
print(sample.shape); print(sample["label"].value_counts(normalize=True))
print(sample.sample(20, random_state=1)[["text", "rating"]].to_string())  # eyeball quality!

from google.colab import files
files.download("banglish_sample.csv")

label
1    115281
0     27160
Name: count, dtype: int64
(15000, 3)
label
1    0.666667
0    0.333333
Name: proportion, dtype: float64
                                                                                                                                                                                                                                                                                                                                           text  rating
7576                                                                                                                                                                                                                                                                                                                          ank valo product.       5
10509                                                                                                                                                                             

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>